# Laboratório IV · Análise: técnica, indicadores, cadeia e ciclos
*Bitcoin na Veia: do halving ao ETF*, 2ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. É o Laboratório 19 da *Análise Técnica dos Mercados Financeiros*, 4ª edição, ampliado. Para o Capítulo 12, testa a escala logarítmica nos fundos de ciclo, o alvo do topo duplo de 2021, a faixa de suporte e resistência dos topos de 2021, a divergência do IFR semanal, a média de 200 dias como filtro e a volatilidade que define o tamanho do stop. Para o Capítulo 13, refaz os indicadores próprios do Bitcoin do Laboratório 19 (ciclo do halving, média de 200 semanas, múltiplo de Mayer, TD Setup, gap da CME e fôlego) e os amplia. Para o Capítulo 14, mede a armadilha das métricas que seguem o preço, o uso da rede, a emissão em dólar contra o fluxo dos ETFs, a liquidez global e a taxa de financiamento como sentimento. Para o Capítulo 15, compara os três ciclos completos, o tempo para recuperar a máxima e os sinais nos topos e nos fundos.

In [ ]:
!pip install -q yfinance

In [2]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
RAW = "https://raw.githubusercontent.com/traderbrasil/livro-bitcoin-na-veia/main/labs/dados/"
def baixa(nome):
    """Usa a cópia dos dados salva no GitHub do livro (mesmos números do livro); sem ela, baixa tudo de novo."""
    if not os.path.exists(nome):
        try:
            import urllib.request
            urllib.request.urlretrieve(RAW + nome, nome)
        except Exception:
            pass
    return os.path.exists(nome)
CACHE = "dados_lab4.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=0: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=0: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
MOEDAS = ["ETH", "XRP", "BNB", "SOL", "DOGE", "ADA", "TRX", "LINK", "XLM", "BCH", "LTC", "AVAX", "DOT",
          "HBAR", "ATOM", "ETC", "XMR", "UNI", "AAVE", "FIL"]      # a mesma lista do Laboratório 19 (UNI-USD não tem dados no Yahoo)
TICKERS = ["BTC-USD", "BTC=F", "IBIT"] + [f"{m}-USD" for m in MOEDAS]
if baixa(CACHE):
    D = pd.read_pickle(CACHE)
else:
    d = yf.download(TICKERS, start="2014-01-01", end=FIM, auto_adjust=True, progress=False)
    D = {k: d[k] for k in ("Open", "High", "Low", "Close")}
    pd.to_pickle(D, CACHE)
CL = D["Close"]
btc = CL["BTC-USD"].dropna(); btc = btc[btc > 0]; hi = D["High"]["BTC-USD"].reindex(btc.index); lo = D["Low"]["BTC-USD"].reindex(btc.index)
SEM = btc.resample("W-SUN").last(); SEM = SEM[SEM.index <= btc.index[-1]]    # semana de cripto: fecha domingo, 23:59 UTC
print(f"Bitcoin de {btc.index[0]:%d/%m/%Y} a {btc.index[-1]:%d/%m/%Y}; último fechamento US$ {nr(btc.iloc[-1])}")

Bitcoin de 17/09/2014 a 30/09/2026; último fechamento US$ 83.554


## Parte 1: escala logarítmica e os fundos de ciclo (Capítulo 12)

In [3]:
JANELAS = {"2015": ("2014-09-17", "2016-01-01"), "2018": ("2018-01-01", "2019-06-01"),
           "2022": ("2022-01-01", "2023-06-01"), "2026": ("2026-01-01", "2026-09-30")}
FUNDOS = {k: btc.loc[a:b].idxmin() for k, (a, b) in JANELAS.items()}
print("Parte 1a. Fundos de ciclo (fechamento diário)")
ant = None
for k, t in FUNDOS.items():
    print(f"   {t:%d/%m/%Y}: US$ {nr(btc[t])}" + (f" | {nr(btc[t] / btc[ant], 1)} vezes o fundo anterior" if ant is not None else ""))
    ant = t
def linha(t0, t1, t):
    y0, y1 = np.log(btc[t0]), np.log(btc[t1]); s = (y1 - y0) / (t1 - t0).days
    return np.exp(y0 + s * (t - t0).days), np.exp(s * 365.25) - 1
print("\nParte 1b. Linhas retas em escala log ligando dois fundos, e onde ficou o fundo seguinte")
for a, b, c in [("2015", "2018", "2022"), ("2018", "2022", "2026")]:
    v, taxa = linha(FUNDOS[a], FUNDOS[b], FUNDOS[c])
    print(f"   Linha {a}-{b} ({ps(taxa)} ao ano): em {FUNDOS[c]:%d/%m/%Y} valia US$ {nr(v)}; o fundo foi US$ {nr(btc[FUNDOS[c]])} ({ps(btc[FUNDOS[c]] / v - 1)})")
print("   Em escala log, a linha dos fundos fica menos inclinada a cada ciclo: os retornos estão diminuindo.")

Parte 1a. Fundos de ciclo (fechamento diário)
   14/01/2015: US$ 178
   15/12/2018: US$ 3.237 | 18,2 vezes o fundo anterior
   21/11/2022: US$ 15.787 | 4,9 vezes o fundo anterior
   30/06/2026: US$ 58.559 | 3,7 vezes o fundo anterior

Parte 1b. Linhas retas em escala log ligando dois fundos, e onde ficou o fundo seguinte
   Linha 2015-2018 (+110% ao ano): em 21/11/2022 valia US$ 59.543; o fundo foi US$ 15.787 (-73%)
   Linha 2018-2022 (+50% ao ano): em 30/06/2026 valia US$ 67.458; o fundo foi US$ 58.559 (-13%)
   Em escala log, a linha dos fundos fica menos inclinada a cada ciclo: os retornos estão diminuindo.


## Parte 2: o topo duplo de 2021 e o alvo em escala log (Capítulo 12)

In [4]:
t1 = btc.loc["2021-03-01":"2021-05-01"]; t2 = btc.loc["2021-10-01":"2021-12-01"]; vale = btc.loc["2021-05-01":"2021-09-01"]
topo = max(t1.max(), t2.max()); pescoco = vale.min()
depois = btc.loc["2021-12-01":]; rompe = depois[depois < pescoco].index[0]
fundo = btc.loc["2022-01-01":"2023-06-01"]
print(f"Parte 2. Topos de {t1.idxmax():%d/%m/%Y} (US$ {nr(t1.max())}) e {t2.idxmax():%d/%m/%Y} (US$ {nr(t2.max())}); linha de pescoço em US$ {nr(pescoco)} ({vale.idxmin():%d/%m/%Y})")
print(f"   Rompimento no fechamento de {rompe:%d/%m/%Y}, a US$ {nr(btc[rompe])}")
print(f"   Alvo na escala aritmética: US$ {nr(pescoco - (topo - pescoco))} (impossível)")
print(f"   Alvo na escala log: US$ {nr(pescoco * pescoco / topo)} | fundo real: US$ {nr(fundo.min())} em {fundo.idxmin():%d/%m/%Y}")

Parte 2. Topos de 13/04/2021 (US$ 63.503) e 08/11/2021 (US$ 67.567); linha de pescoço em US$ 29.807 (20/07/2021)
   Rompimento no fechamento de 11/05/2022, a US$ 28.936
   Alvo na escala aritmética: US$ -7.952 (impossível)
   Alvo na escala log: US$ 13.150 | fundo real: US$ 15.787 em 21/11/2022


## Parte 3: a faixa dos topos de 2021 como resistência e suporte (Capítulo 12)

In [5]:
BAIXO, ALTO = 63_500, 69_000
s = SEM.loc["2024-01-01":]
print(f"Parte 3. Faixa de US$ {nr(BAIXO)} a {nr(ALTO)} (topos de abr e nov/2021), fechamentos semanais desde 2024")
print(f"   Primeiro fechamento semanal acima da faixa: {s[s > ALTO].index[0]:%d/%m/%Y}")
mar = btc.loc["2024-03"]; faixa = btc.loc["2024-03-15":"2024-11-05"]
print(f"   Máxima de {mar.idxmax():%d/%m/%Y}: US$ {nr(mar.max())}; depois, lateral entre US$ {nr(faixa.min())} e {nr(faixa.max())} até nov/2024")
r = btc.loc["2024-03-15":]; r = r[r > mar.max()].index[0]
print(f"   Rompimento da máxima de março em {r:%d/%m/%Y}, a US$ {nr(btc[r])}")
s26 = SEM.loc["2026-01-01":]
dentro = s26[(s26 >= BAIXO) & (s26 <= ALTO)]; abaixo = s26[s26 < BAIXO]
print(f"   Em 2026: {len(dentro)} fechamentos semanais dentro da faixa e {len(abaixo)} abaixo dela"
      f" (o primeiro em {abaixo.index[0]:%d/%m/%Y}; o menor, US$ {nr(abaixo.min())} em {abaixo.idxmin():%d/%m/%Y})")
volta = s26.loc[abaixo.index[-1]:]; volta = volta[volta > ALTO]
print(f"   Volta para cima da faixa: semana de {volta.index[0]:%d/%m/%Y}, a US$ {nr(volta.iloc[0])}")
print("   A antiga resistência segurou como suporte de fevereiro a maio, cedeu em junho e foi recuperada em agosto.")

Parte 3. Faixa de US$ 63.500 a 69.000 (topos de abr e nov/2021), fechamentos semanais desde 2024
   Primeiro fechamento semanal acima da faixa: 10/03/2024
   Máxima de 13/03/2024: US$ 73.084; depois, lateral entre US$ 53.949 e 72.720 até nov/2024
   Rompimento da máxima de março em 06/11/2024, a US$ 75.639
   Em 2026: 13 fechamentos semanais dentro da faixa e 5 abaixo dela (o primeiro em 07/06/2026; o menor, US$ 59.532 em 28/06/2026)
   Volta para cima da faixa: semana de 23/08/2026, a US$ 77.755
   A antiga resistência segurou como suporte de fevereiro a maio, cedeu em junho e foi recuperada em agosto.


## Parte 4: o IFR semanal nos topos e fundos (Capítulo 12)

In [6]:
def ifr(s, n=14):
    d = s.diff(); sobe = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean(); cai = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + sobe / cai)
IFR = ifr(SEM)
PONTOS = [("topo de 2017", "2017-11-01", "2018-01-31", "max"), ("fundo de 2018", "2018-11-01", "2019-01-31", "min"),
          ("topo de abr/2021", "2021-03-01", "2021-05-15", "max"), ("topo de nov/2021", "2021-10-15", "2021-11-30", "max"),
          ("fundo de 2022", "2022-06-01", "2022-12-31", "min"), ("topo de mar/2024", "2024-03-01", "2024-03-31", "max"),
          ("topo de dez/2024", "2024-12-01", "2024-12-31", "max"), ("topo de out/2025", "2025-09-25", "2025-10-20", "max"),
          ("fundo de 2026", "2026-02-01", "2026-07-15", "min")]
print("Parte 4. Fechamento semanal e IFR semanal (14) nos pontos extremos")
for nome, a, b, tipo in PONTOS:
    w = SEM.loc[a:b]; t = w.idxmax() if tipo == "max" else w.idxmin(); r = IFR.loc[a:b]
    print(f"   {nome:<17} semana de {t:%d/%m/%Y}: US$ {nr(w[t]):>7} | IFR {nr(IFR[t], 1)} (extremo do IFR na janela: {nr(r.max() if tipo == 'max' else r.min(), 1)})")
print("   2024-2025: o preço fez topos mais altos (mar/2024, dez/2024, out/2025) com IFR mais baixo a cada vez: divergência de baixa.")

Parte 4. Fechamento semanal e IFR semanal (14) nos pontos extremos
   topo de 2017      semana de 17/12/2017: US$  19.141 | IFR 90,3 (extremo do IFR na janela: 90,3)
   fundo de 2018     semana de 16/12/2018: US$   3.253 | IFR 28,8 (extremo do IFR na janela: 28,8)
   topo de abr/2021  semana de 11/04/2021: US$  60.205 | IFR 73,0 (extremo do IFR na janela: 75,1)
   topo de nov/2021  semana de 14/11/2021: US$  65.467 | IFR 68,8 (extremo do IFR na janela: 68,8)
   fundo de 2022     semana de 20/11/2022: US$  16.292 | IFR 31,4 (extremo do IFR na janela: 25,7)
   topo de mar/2024  semana de 31/03/2024: US$  71.334 | IFR 85,6 (extremo do IFR na janela: 88,3)
   topo de dez/2024  semana de 15/12/2024: US$ 104.299 | IFR 78,7 (extremo do IFR na janela: 78,7)
   topo de out/2025  semana de 05/10/2025: US$ 123.513 | IFR 64,2 (extremo do IFR na janela: 64,2)
   fundo de 2026     semana de 28/06/2026: US$  59.532 | IFR 32,9 (extremo do IFR na janela: 26,8)
   2024-2025: o preço fez topos mais altos

## Parte 5: a média de 200 dias como filtro de tendência (Capítulo 12)

In [7]:
CUSTO = 0.002                                   # 0,2% por troca de posição (hipótese)
mm = btc.rolling(200).mean()
pos = (btc > mm).astype(int).shift(1)          # decide no fechamento, opera no dia seguinte
r = btc.pct_change()
df = pd.DataFrame({"r": r, "pos": pos}).dropna().loc["2015-04-01":]
trocas = df["pos"].diff().abs().fillna(0)
regra = (1 + df["pos"] * df["r"] - trocas * CUSTO).cumprod(); seg = (1 + df["r"]).cumprod()
anos = (df.index[-1] - df.index[0]).days / 365.25
qd = lambda e: (e / e.cummax() - 1).min()
print(f"Parte 5. Comprado só quando o fechamento está acima da média de 200 dias, de {df.index[0]:%d/%m/%Y} a 30/09/2026, custo de {pc(CUSTO, 1)} por troca")
print(f"   Regra: {pc(regra.iloc[-1] ** (1 / anos) - 1, 1)} ao ano | maior queda {pc(qd(regra))} | tempo comprado {pc(df['pos'].mean())} | {int(trocas.sum())} trocas")
print(f"   Comprar e segurar: {pc(seg.iloc[-1] ** (1 / anos) - 1, 1)} ao ano | maior queda {pc(qd(seg))}")
ano = df.groupby(df.index.year).apply(lambda x: pd.Series({"segurar": (1 + x["r"]).prod() - 1, "regra": (1 + x["pos"] * x["r"]).prod() - 1}))
print("   Por ano (sem custo):")
for a, l in ano.iterrows():
    print(f"   {a}: segurar {ps(l['segurar']):>7} | regra {ps(l['regra']):>7}")
print("   A média reduziu a queda, mas custou retorno e trocou de lado muitas vezes nas laterais. Um teste no passado não é promessa.")

Parte 5. Comprado só quando o fechamento está acima da média de 200 dias, de 01/04/2015 a 30/09/2026, custo de 0,2% por troca
   Regra: 59,3% ao ano | maior queda -70% | tempo comprado 61% | 75 trocas
   Comprar e segurar: 66,1% ao ano | maior queda -83%
   Por ano (sem custo):
   2015: segurar    +76% | regra    +39%
   2016: segurar   +124% | regra   +124%
   2017: segurar  +1369% | regra  +1369%
   2018: segurar    -74% | regra    -59%
   2019: segurar    +92% | regra    +58%
   2020: segurar   +303% | regra   +183%
   2021: segurar    +60% | regra    -20%
   2022: segurar    -64% | regra     +0%
   2023: segurar   +155% | regra    +95%
   2024: segurar   +121% | regra    +83%
   2025: segurar     -6% | regra    -16%
   2026: segurar     -5% | regra    +21%
   A média reduziu a queda, mas custou retorno e trocou de lado muitas vezes nas laterais. Um teste no passado não é promessa.


## Parte 6: volatilidade e o tamanho do stop (Capítulo 12)

In [8]:
amp = (hi / lo - 1).dropna()
tr = pd.concat([hi - lo, (hi - btc.shift()).abs(), (lo - btc.shift()).abs()], axis=1).max(axis=1)
atr = (tr.rolling(14).mean() / btc).dropna()
print("Parte 6. Amplitude diária (máxima / mínima − 1) e ATR de 14 dias em % do preço")
for a in [2015, 2017, 2019, 2021, 2023, 2024, 2025, 2026]:
    x = amp[str(a)]
    print(f"   {a}: amplitude média {pc(x.mean(), 1)} | dias com amplitude acima de 5%: {pc((x > 0.05).mean())} | ATR mediano {pc(atr[str(a)].median(), 1)}")
print(f"   ATR em 30/09/2026: {pc(atr.iloc[-1], 2)}. Um stop de 2 ATR ficaria {pc(2 * atr.iloc[-1], 1)} abaixo do preço.")
print("   A amplitude passou de 5% em mais da metade dos dias de 2017; em 2026, em um de cada sete: um stop fixo de 5% vira ruído.")

Parte 6. Amplitude diária (máxima / mínima − 1) e ATR de 14 dias em % do preço
   2015: amplitude média 4,2% | dias com amplitude acima de 5%: 24% | ATR mediano 3,5%
   2017: amplitude média 6,8% | dias com amplitude acima de 5%: 53% | ATR mediano 5,6%
   2019: amplitude média 4,2% | dias com amplitude acima de 5%: 25% | ATR mediano 3,6%
   2021: amplitude média 6,7% | dias com amplitude acima de 5%: 59% | ATR mediano 5,7%
   2023: amplitude média 3,1% | dias com amplitude acima de 5%: 15% | ATR mediano 2,9%
   2024: amplitude média 4,2% | dias com amplitude acima de 5%: 28% | ATR mediano 3,8%
   2025: amplitude média 3,4% | dias com amplitude acima de 5%: 17% | ATR mediano 3,1%
   2026: amplitude média 3,4% | dias com amplitude acima de 5%: 15% | ATR mediano 3,0%
   ATR em 30/09/2026: 2,83%. Um stop de 2 ATR ficaria 5,7% abaixo do preço.
   A amplitude passou de 5% em mais da metade dos dias de 2017; em 2026, em um de cada sete: um stop fixo de 5% vira ruído.


## Parte 7: o ciclo do halving (Capítulo 13)

In [9]:
HALVINGS = ["2016-07-09", "2020-05-11", "2024-04-20"]     # o de 28/11/2012 é anterior aos dados do Yahoo
print("Parte 7. Bitcoin depois de cada halving (mesmo cálculo do Laboratório 19)")
for h in map(pd.Timestamp, HALVINGS):
    x = btc.loc[h:h + pd.Timedelta(days=730)]; topo = x.idxmax()
    depois = btc.loc[topo:topo + pd.Timedelta(days=500)]
    print(f"   halving {h:%d/%m/%Y}: US$ {nr(x.iloc[0])} | topo em {topo:%d/%m/%Y} ({(topo - h).days} dias depois), US$ {nr(x.max())}"
          f" ({nr(x.max() / x.iloc[0], 1)} vezes) | maior queda depois: {pc(depois.min() / x.max() - 1)} (mínima em {depois.idxmin():%d/%m/%Y},"
          f" {(depois.idxmin() - topo).days} dias depois do topo)")
print("   O topo chegou entre 525 e 546 dias depois do halving, mas a alta encolheu de 30 para 1,9 vezes. O ciclo de 2024 ainda não acabou em 30/09/2026.")

Parte 7. Bitcoin depois de cada halving (mesmo cálculo do Laboratório 19)
   halving 09/07/2016: US$ 651 | topo em 16/12/2017 (525 dias depois), US$ 19.497 (30,0 vezes) | maior queda depois: -83% (mínima em 15/12/2018, 364 dias depois do topo)
   halving 11/05/2020: US$ 8.602 | topo em 08/11/2021 (546 dias depois), US$ 67.567 (7,9 vezes) | maior queda depois: -77% (mínima em 21/11/2022, 378 dias depois do topo)
   halving 20/04/2024: US$ 64.994 | topo em 06/10/2025 (534 dias depois), US$ 124.753 (1,9 vezes) | maior queda depois: -53% (mínima em 30/06/2026, 267 dias depois do topo)
   O topo chegou entre 525 e 546 dias depois do halving, mas a alta encolheu de 30 para 1,9 vezes. O ciclo de 2024 ainda não acabou em 30/09/2026.


## Parte 8: a média de 200 semanas (Capítulo 13)

In [10]:
SEM19 = btc.resample("W-SUN").last()        # como no Laboratório 19, a última semana vai até 30/09/2026
MM200S = SEM19.rolling(200).mean(); ABAIXO = (SEM19 < MM200S)[MM200S.notna()]
print(f"Parte 8. Média de 200 semanas em 30/09/2026: US$ {nr(MM200S.iloc[-1])}; preço {nr(btc.iloc[-1] / MM200S.iloc[-1], 2)} vez a média")
print(f"   Fechamentos semanais abaixo dela desde {MM200S.dropna().index[0]:%Y}: {pc(ABAIXO.mean())} ({ABAIXO.sum()} de {len(ABAIXO)} semanas)")
for nome, a, b in [("fundo de 2018", "2018-11-01", "2019-02-28"), ("Covid, 2020", "2020-03-01", "2020-04-30"),
                   ("fundo de 2022", "2022-06-01", "2022-12-31"), ("fundo de 2026", "2026-01-01", "2026-09-30")]:
    r = (SEM19 / MM200S).loc[a:b]
    print(f"   {nome:<14} menor razão preço / média: {nr(r.min(), 2)} na semana de {r.idxmin():%d/%m/%Y}")

Parte 8. Média de 200 semanas em 30/09/2026: US$ 66.166; preço 1,26 vez a média


   Fechamentos semanais abaixo dela desde 2018: 12% (50 de 430 semanas)
   fundo de 2018  menor razão preço / média: 1,02 na semana de 16/12/2018
   Covid, 2020    menor razão preço / média: 0,97 na semana de 15/03/2020
   fundo de 2022  menor razão preço / média: 0,68 na semana de 20/11/2022
   fundo de 2026  menor razão preço / média: 0,95 na semana de 28/06/2026


## Parte 9: o múltiplo de Mayer (Capítulo 13)

In [11]:
MAYER = btc / btc.rolling(200).mean()
F365 = btc.shift(-365) / btc - 1
T4 = pd.DataFrame({"m": MAYER, "f": F365}).dropna()
print(f"Parte 9. Múltiplo de Mayer em 30/09/2026: {nr(MAYER.iloc[-1], 2)}. Bitcoin um ano depois, por faixa (desde 2015, como no Laboratório 19):")
for nome, cond in [("abaixo de 0,8", T4["m"] < 0.8), ("0,8 a 1,5", T4["m"].between(0.8, 1.5)),
                   ("1,5 a 2,4", T4["m"].between(1.5, 2.4, inclusive="right")), ("acima de 2,4", T4["m"] > 2.4)]:
    x = T4.loc[cond, "f"]
    print(f"   {nome:<14} {nr(len(x)):>5} dias | mediana {ps(x.median())} | positivo em {pc((x > 0).mean())}")
print("   O pico do múltiplo em cada alta:")
for nome, a, b in [("2017", "2017-01-01", "2018-06-30"), ("2020-2021", "2020-06-01", "2021-06-30"), ("nov/2021", "2021-07-01", "2021-12-31"),
                   ("2024", "2023-01-01", "2024-06-30"), ("2024-2025", "2024-07-01", "2025-12-31")]:
    m = MAYER.loc[a:b]
    print(f"   {nome:<10} {nr(m.max(), 2)} em {m.idxmax():%d/%m/%Y}")
print(f"   No topo de 06/10/2025: {nr(MAYER['2025-10-06'], 2)}. Última vez acima de 2,4: {MAYER[MAYER > 2.4].index[-1]:%d/%m/%Y}.")
print("   Os picos encolheram a cada ciclo. O último sinal acima de 2,4 foi um mês antes do topo de abr/2021; nos topos de nov/2021, 2024 e 2025, o múltiplo nem chegou perto.")

Parte 9. Múltiplo de Mayer em 30/09/2026: 1,17. Bitcoin um ano depois, por faixa (desde 2015, como no Laboratório 19):
   abaixo de 0,8    547 dias | mediana +50% | positivo em 78%
   0,8 a 1,5      2.662 dias | mediana +102% | positivo em 73%
   1,5 a 2,4        563 dias | mediana +25% | positivo em 71%
   acima de 2,4      61 dias | mediana -66% | positivo em 15%
   O pico do múltiplo em cada alta:
   2017       3,93 em 07/12/2017
   2020-2021  2,83 em 08/01/2021
   nov/2021   1,48 em 08/11/2021
   2024       1,85 em 13/03/2024
   2024-2025  1,53 em 16/12/2024
   No topo de 06/10/2025: 1,18. Última vez acima de 2,4: 13/03/2021.
   Os picos encolheram a cada ciclo. O último sinal acima de 2,4 foi um mês antes do topo de abr/2021; nos topos de nov/2021, 2024 e 2025, o múltiplo nem chegou perto.


## Parte 10: o TD Setup (Capítulo 13)

In [12]:
compra = (btc < btc.shift(4)).astype(int); venda = (btc > btc.shift(4)).astype(int)
CONT_C = compra.groupby((compra == 0).cumsum()).cumsum(); CONT_V = venda.groupby((venda == 0).cumsum()).cumsum()
print("Parte 10. TD Setup (9) no Bitcoin diário: retorno médio depois do sinal (entre parênteses, % de altas)")
for nome, cond in [("9 de compra", CONT_C == 9), ("9 de venda", CONT_V == 9), ("qualquer dia", btc.notna())]:
    txt = f"   {nome:<13} {int(cond.sum()):>5} sinais"
    for h in (5, 10, 20):
        x = (btc.shift(-h) / btc - 1)[cond].dropna(); txt += f" | {h}d {ps(x.mean(), 1)} ({pc((x > 0).mean())})"
    print(txt)
mm = btc.rolling(200).mean()
for nome, cond in [("9 de compra acima da média de 200 dias", (CONT_C == 9) & (btc > mm)), ("9 de compra abaixo da média de 200 dias", (CONT_C == 9) & (btc < mm))]:
    x = (btc.shift(-20) / btc - 1)[cond].dropna()
    print(f"   {nome}: {len(x)} sinais | 20d {ps(x.mean(), 1)} ({pc((x > 0).mean())})")
print("   Poucos sinais: 45 de compra em 12 anos. A vantagem existe na média, mas cada sinal isolado erra muito.")

Parte 10. TD Setup (9) no Bitcoin diário: retorno médio depois do sinal (entre parênteses, % de altas)
   9 de compra      45 sinais | 5d +3,2% (60%) | 10d +5,6% (73%) | 20d +5,8% (60%)
   9 de venda       75 sinais | 5d +2,1% (51%) | 10d +4,1% (64%) | 20d +9,8% (64%)
   qualquer dia   4397 sinais | 5d +0,9% (54%) | 10d +1,8% (56%) | 20d +3,9% (56%)
   9 de compra acima da média de 200 dias: 18 sinais | 20d +6,6% (50%)
   9 de compra abaixo da média de 200 dias: 25 sinais | 20d +4,9% (64%)
   Poucos sinais: 45 de compra em 12 anos. A vantagem existe na média, mas cada sinal isolado erra muito.


## Parte 11: o gap da CME e o gap do ETF (Capítulo 13)

In [13]:
def gaps(df):
    seg = df.index.to_series().diff().dt.days >= 3          # primeira sessão depois do fim de semana (ou feriado)
    G = []
    for i in np.where(seg.values)[0]:
        ant, hoje = df.iloc[i - 1], df.iloc[i]; nivel = ant["Close"]
        if hoje["Low"] > nivel: tipo = "alta"
        elif hoje["High"] < nivel: tipo = "baixa"
        else: continue
        prox = df.iloc[i + 1:i + 21]; toca = (prox["Low"] <= nivel) if tipo == "alta" else (prox["High"] >= nivel)
        G.append((df.index[i], tipo, abs(hoje["Open"] / nivel - 1), int(np.argmax(toca.values)) + 1 if toca.any() else np.nan))
    return pd.DataFrame(G, columns=["data", "tipo", "tam", "dias"]), int(seg.sum())
OHLC = lambda t: pd.DataFrame({k: D[k][t] for k in ("Open", "High", "Low", "Close")}).dropna()
FUT = OHLC("BTC=F"); FUT = FUT[FUT.index < "2026-05-29"]   # a partir de 29/05/2026 a CME negocia cripto 24 horas por dia
for nome, df in [("Futuro da CME (dez/2017 a mai/2026)", FUT), ("IBIT (jan/2024 a set/2026)", OHLC("IBIT"))]:
    G, n = gaps(df)
    print(f"Parte 11. {nome}: {len(G)} gaps de fim de semana não fechados no primeiro pregão, de {n} recomeços de semana")
    for t, x in [("todos", G), ("de alta", G[G["tipo"] == "alta"]), ("de baixa", G[G["tipo"] == "baixa"])]:
        print(f"   {t:<9} {len(x):>4} | tamanho mediano {pc(x['tam'].median(), 1)} | fechados em 5 pregões: {pc((x['dias'] <= 5).mean())}"
              f" | em 20: {pc((x['dias'] <= 20).mean())}")
print("   Com a CME aberta 24 horas desde 29/05/2026, o gap de fim de semana migrou para os ETFs, que só negociam em dias úteis.")

Parte 11. Futuro da CME (dez/2017 a mai/2026): 194 gaps de fim de semana não fechados no primeiro pregão, de 440 recomeços de semana


   todos      194 | tamanho mediano 2,2% | fechados em 5 pregões: 60% | em 20: 75%
   de alta    113 | tamanho mediano 2,5% | fechados em 5 pregões: 56% | em 20: 73%
   de baixa    81 | tamanho mediano 2,0% | fechados em 5 pregões: 67% | em 20: 79%
Parte 11. IBIT (jan/2024 a set/2026): 98 gaps de fim de semana não fechados no primeiro pregão, de 142 recomeços de semana
   todos       98 | tamanho mediano 2,4% | fechados em 5 pregões: 63% | em 20: 82%
   de alta     54 | tamanho mediano 2,8% | fechados em 5 pregões: 65% | em 20: 78%
   de baixa    44 | tamanho mediano 2,0% | fechados em 5 pregões: 61% | em 20: 86%
   Com a CME aberta 24 horas desde 29/05/2026, o gap de fim de semana migrou para os ETFs, que só negociam em dias úteis.


## Parte 12: o fôlego cripto (Capítulo 13)

In [14]:
ALT = CL[[f"{m}-USD" for m in MOEDAS]]; ALT = ALT.where(ALT > 0)
MMA = ALT.rolling(200, min_periods=200).mean()
ACIMA = (ALT > MMA).where(MMA.notna() & ALT.notna())
PCT = ACIMA.mean(axis=1)[ACIMA.notna().sum(axis=1) >= 15]
VAR90 = ALT.div(btc, axis=0).pct_change(90, fill_method=None)
MELHOR = (VAR90 > 0).where(VAR90.notna()).mean(axis=1)[VAR90.notna().sum(axis=1) >= 15]
print(f"Parte 12. Fôlego das moedas grandes do Laboratório 19 ({ALT.notna().any().sum()} com dados; só as que sobreviveram)")
print(f"   Em {PCT.index[-1]:%d/%m/%Y}: {pc(PCT.iloc[-1])} acima da média de 200 dias | {pc(MELHOR.iloc[-1])} melhores que o Bitcoin em 90 dias")
for ano in range(max(PCT.index[0].year, MELHOR.index[0].year), 2027):
    print(f"   {ano}: média de {pc(PCT.loc[str(ano)].mean())} acima da média de 200 dias | {pc(MELHOR.loc[str(ano)].mean())} batendo o Bitcoin em 90 dias")

Parte 12. Fôlego das moedas grandes do Laboratório 19 (19 com dados; só as que sobreviveram)
   Em 30/09/2026: 95% acima da média de 200 dias | 68% melhores que o Bitcoin em 90 dias
   2020: média de 72% acima da média de 200 dias | 47% batendo o Bitcoin em 90 dias
   2021: média de 70% acima da média de 200 dias | 53% batendo o Bitcoin em 90 dias
   2022: média de 9% acima da média de 200 dias | 41% batendo o Bitcoin em 90 dias
   2023: média de 54% acima da média de 200 dias | 23% batendo o Bitcoin em 90 dias
   2024: média de 58% acima da média de 200 dias | 34% batendo o Bitcoin em 90 dias
   2025: média de 47% acima da média de 200 dias | 37% batendo o Bitcoin em 90 dias
   2026: média de 17% acima da média de 200 dias | 27% batendo o Bitcoin em 90 dias


## Parte 13: dados na cadeia e a armadilha do preço (Capítulo 14)

In [15]:
import io, json, urllib.request, zipfile
CCACHE = "dados_cadeia.pkl"
def blockchain_com(nome):
    url = f"https://api.blockchain.info/charts/{nome}?timespan=all&format=json&sampled=false"
    for tentativa in range(5):                                   # o servidor às vezes demora ou derruba a conexão
        try:
            v = json.loads(urllib.request.urlopen(url, timeout=60).read())["values"]; break
        except Exception:
            if tentativa == 4: raise
    return pd.Series({pd.to_datetime(x["x"], unit="s"): x["y"] for x in v}).loc[:"2026-09-30"]
def fred(serie):                                                # Federal Reserve de St. Louis, sem chave
    txt = urllib.request.urlopen(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={serie}", timeout=60).read().decode()
    d = pd.read_csv(io.StringIO(txt)); d.columns = ["data", "v"]
    return pd.to_numeric(d["v"], errors="coerce").set_axis(pd.to_datetime(d["data"])).loc[:"2026-09-30"]
SERIES = {"hash": "hash-rate", "taxas_btc": "transaction-fees", "taxas_usd": "transaction-fees-usd", "transacoes": "n-transactions",
          "enderecos": "n-unique-addresses", "volume_usd": "estimated-transaction-volume-usd", "receita_usd": "miners-revenue",
          "confirmacao": "median-confirmation-time", "bloco_mb": "avg-block-size"}
if baixa(CCACHE):
    X = pd.read_pickle(CCACHE)
else:
    X = {"cadeia": pd.DataFrame({k: blockchain_com(v) for k, v in SERIES.items()}), "m2": fred("M2SL"), "walcl": fred("WALCL")}
    pd.to_pickle(X, CCACHE)
CAD = X["cadeia"]
MES = CAD.loc["2015":].resample("ME").mean(); MES["preco"] = btc.resample("ME").mean()
VAR = np.log(MES).diff().dropna()
print("Parte 13. Correlação das variações mensais de cada métrica com as do preço (médias mensais desde 2015)")
for k, nome in [("receita_usd", "receita dos mineradores, em dólar"), ("volume_usd", "volume transferido, em dólar"),
                ("taxas_usd", "taxas pagas, em dólar"), ("enderecos", "endereços ativos por dia"), ("hash", "taxa de hash"),
                ("taxas_btc", "taxas pagas, em bitcoins"), ("transacoes", "número de transações"), ("bloco_mb", "tamanho médio do bloco")]:
    print(f"   {nome:<36} {nr(VAR['preco'].corr(VAR[k]), 2):>5}")
print("   As métricas em dólar contam a história do preço, e não a do uso. Taxas em dólar, bitcoins e transações: os mesmos +0,51, +0,24 e +0,21 do Laboratório 19.")

Parte 13. Correlação das variações mensais de cada métrica com as do preço (médias mensais desde 2015)
   receita dos mineradores, em dólar     0,89
   volume transferido, em dólar          0,64
   taxas pagas, em dólar                 0,51
   endereços ativos por dia              0,50
   taxa de hash                          0,29
   taxas pagas, em bitcoins              0,24
   número de transações                  0,21
   tamanho médio do bloco                0,20
   As métricas em dólar contam a história do preço, e não a do uso. Taxas em dólar, bitcoins e transações: os mesmos +0,51, +0,24 e +0,21 do Laboratório 19.


## Parte 14: o uso da rede, de 2017 a 2026 (Capítulo 14)

In [16]:
U = CAD.assign(taxa_media=CAD["taxas_usd"] / CAD["transacoes"], custo_total=CAD["receita_usd"] / CAD["transacoes"])
print("Parte 14. Médias de cada ano")
for a in [2017, 2019, 2021, 2023, 2024, 2025, 2026]:
    m = U.loc[str(a)].mean()
    print(f"   {a}{'*' if a == 2026 else ' '}: {nr(m['transacoes']):>7} transações/dia | {nr(m['enderecos']):>7} endereços/dia | bloco {nr(m['bloco_mb'], 2)} MB"
          f" | confirmação {nr(m['confirmacao'], 1)} min | taxa média US$ {nr(m['taxa_media'], 2)} | custo total por transação US$ {nr(m['custo_total'], 0)}")
print("   * 2026 até 30/09. Custo total = receita dos mineradores (recompensa + taxas) dividida pelas transações: é o que a rede gasta para registrar cada uma.")

Parte 14. Médias de cada ano
   2017 : 285.105 transações/dia | 569.236 endereços/dia | bloco 0,95 MB | confirmação 13,0 min | taxa média US$ 4,89 | custo total por transação US$ 31
   2019 : 328.174 transações/dia | 530.164 endereços/dia | bloco 1,07 MB | confirmação 8,2 min | taxa média US$ 1,25 | custo total por transação US$ 44
   2021 : 267.935 transações/dia | 680.251 endereços/dia | bloco 1,22 MB | confirmação 9,2 min | taxa média US$ 9,98 | custo total por transação US$ 173
   2023 : 420.318 transações/dia | 694.757 endereços/dia | bloco 1,70 MB | confirmação 9,0 min | taxa média US$ 4,38 | custo total por transação US$ 71
   2024 : 525.269 transações/dia | 552.891 endereços/dia | bloco 1,66 MB | confirmação 9,7 min | taxa média US$ 5,20 | custo total por transação US$ 87
   2025 : 420.660 transações/dia | 525.432 endereços/dia | bloco 1,57 MB | confirmação 7,8 min | taxa média US$ 1,19 | custo total por transação US$ 116
   2026*: 580.240 transações/dia | 479.274 endereços/dia

## Parte 15: a emissão em dólar e o fluxo dos ETFs (Capítulo 14)

In [17]:
print("Parte 15. Bitcoins novos por dia vezes o preço médio do ano: o dinheiro novo necessário para o preço não cair")
for a, rec in [(2017, 12.5), (2021, 6.25), (2025, 3.125), (2026, 3.125)]:
    p = btc[str(a)].mean()
    print(f"   {a}: {nr(rec * 144)} BTC por dia x US$ {nr(p)} = US$ {nr(rec * 144 * p / 1e6, 1)} milhões por dia")
ENTRADAS, PREGOES = 55.161e9, 674          # entradas líquidas nos ETFs americanos de 11/01/2024 a 18/09/2026 e pregões no período
print(f"   ETFs de bitcoin nos EUA: US$ {nr(ENTRADAS / 1e9, 1)} bilhões líquidos em {PREGOES} pregões = US$ {nr(ENTRADAS / PREGOES / 1e6, 1)} milhões por pregão")
print("   Por semana: cerca de US$ 410 milhões de entradas nos ETFs contra cerca de US$ 230 a 320 milhões de emissão. O fluxo pesa mais que o halving.")

Parte 15. Bitcoins novos por dia vezes o preço médio do ano: o dinheiro novo necessário para o preço não cair
   2017: 1.800 BTC por dia x US$ 4.006 = US$ 7,2 milhões por dia
   2021: 900 BTC por dia x US$ 47.437 = US$ 42,7 milhões por dia
   2025: 450 BTC por dia x US$ 101.642 = US$ 45,7 milhões por dia
   2026: 450 BTC por dia x US$ 73.088 = US$ 32,9 milhões por dia
   ETFs de bitcoin nos EUA: US$ 55,2 bilhões líquidos em 674 pregões = US$ 81,8 milhões por pregão
   Por semana: cerca de US$ 410 milhões de entradas nos ETFs contra cerca de US$ 230 a 320 milhões de emissão. O fluxo pesa mais que o halving.


## Parte 16: liquidez global, o combustível que não aparece sozinho (Capítulo 14)

In [18]:
mes = btc.resample("ME").last()
LIQ = pd.DataFrame({"btc": mes.pct_change(12), "m2": X["m2"].resample("ME").last().pct_change(12),
                    "fed": X["walcl"].resample("ME").last().pct_change(12)}).dropna().loc["2016":]
print("Parte 16. Correlação da variação em 12 meses do bitcoin com a do M2 americano e a do balanço do Fed")
print(f"   2016 a 2026: M2 {nr(LIQ['btc'].corr(LIQ['m2']), 2)} | balanço do Fed {nr(LIQ['btc'].corr(LIQ['fed']), 2)}")
for a, b in [("2016", "2019"), ("2020", "2022"), ("2023", "2026")]:
    x = LIQ.loc[a:b]
    print(f"   {a} a {b}: M2 {nr(x['btc'].corr(x['m2']), 2):>5} | balanço do Fed {nr(x['btc'].corr(x['fed']), 2):>5}")
print("   A relação muda de sinal de um período para outro. De 2023 a 2026, o Fed encolheu o balanço e o bitcoin subiu: olhar só o Fed engana.")

Parte 16. Correlação da variação em 12 meses do bitcoin com a do M2 americano e a do balanço do Fed
   2016 a 2026: M2 0,17 | balanço do Fed 0,06
   2016 a 2019: M2 -0,09 | balanço do Fed  0,46
   2020 a 2022: M2  0,48 | balanço do Fed  0,13
   2023 a 2026: M2 -0,08 | balanço do Fed -0,85
   A relação muda de sinal de um período para outro. De 2023 a 2026, o Fed encolheu o balanço e o bitcoin subiu: olhar só o Fed engana.


## Parte 17: a taxa de financiamento como sentimento (Capítulo 14)

In [19]:
FCACHE = "funding_btcusdt.pkl"
if baixa(FCACHE):
    F = pd.read_pickle(FCACHE)
else:   # os mesmos arquivos públicos da Binance do Laboratório III, Parte 7
    partes = []
    for per in pd.period_range("2020-01", "2026-09", freq="M"):
        u = f"https://data.binance.vision/data/futures/um/monthly/fundingRate/BTCUSDT/BTCUSDT-fundingRate-{per.year}-{per.month:02d}.zip"
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(u, timeout=60).read())); partes.append(pd.read_csv(z.open(z.namelist()[0])))
    F = pd.concat(partes); F.to_pickle(FCACHE)
fr = F.assign(t=pd.to_datetime(F["calc_time"], unit="ms")).set_index("t").sort_index()["last_funding_rate"]
f30 = fr.resample("D").sum().rolling(30).mean() * 365          # média de 30 dias, anualizada
T = pd.DataFrame({"f": f30, "r": btc.shift(-90) / btc - 1}).dropna()
q10, q90 = T["f"].quantile(0.1), T["f"].quantile(0.9)
print(f"Parte 17. Taxa de financiamento anualizada (média de 30 dias) e o bitcoin 90 dias depois, 2020 a 2026")
for nome, c in [(f"10% dos dias com taxa mais baixa (até {pc(q10, 1)} ao ano)", T["f"] <= q10), ("os 80% do meio", (T["f"] > q10) & (T["f"] < q90)),
                (f"10% dos dias com taxa mais alta (acima de {pc(q90)} ao ano)", T["f"] >= q90)]:
    x = T.loc[c, "r"]
    print(f"   {nome:<52} {len(x):>5} dias | mediana {ps(x.median())} | em alta {pc((x > 0).mean())}")
print("   Taxa baixa, pessimismo: o bitcoin subiu mais depois. Taxa alta, euforia: subiu menos e caiu mais vezes. Dias sobrepostos inflam a amostra.")

Parte 17. Taxa de financiamento anualizada (média de 30 dias) e o bitcoin 90 dias depois, 2020 a 2026
   10% dos dias com taxa mais baixa (até 1,2% ao ano)     235 dias | mediana +29% | em alta 64%
   os 80% do meio                                        1876 dias | mediana +7% | em alta 60%
   10% dos dias com taxa mais alta (acima de 31% ao ano)   235 dias | mediana -8% | em alta 38%
   Taxa baixa, pessimismo: o bitcoin subiu mais depois. Taxa alta, euforia: subiu menos e caiu mais vezes. Dias sobrepostos inflam a amostra.


## Parte 18: três ciclos completos (Capítulo 15)

In [20]:
FUNDOS_C = [pd.Timestamp(x) for x in ["2015-01-14", "2018-12-15", "2022-11-21", "2026-06-30"]]   # fundos da Parte 1
TOPOS_C = [pd.Timestamp(x) for x in ["2017-12-16", "2021-11-08", "2025-10-06"]]               # topos da Parte 7
print("Parte 18. Do fundo ao topo e do topo ao fundo seguinte (fechamento diário)")
for f, t, f2 in zip(FUNDOS_C[:-1], TOPOS_C, FUNDOS_C[1:]):
    print(f"   {f:%d/%m/%Y} US$ {nr(btc[f])} -> topo {t:%d/%m/%Y} US$ {nr(btc[t])}: {nr(btc[t] / btc[f], 1)} vezes em {(t - f).days} dias"
          f" | -> fundo {f2:%d/%m/%Y} US$ {nr(btc[f2])}: {pc(btc[f2] / btc[t] - 1)} em {(f2 - t).days} dias")
print("   Do fundo ao topo: 1.067, 1.059 e 1.050 dias. A alta encolheu de 109,5 para 20,9 e 7,9 vezes. O fundo de 2026 vale até 30/09/2026.")

Parte 18. Do fundo ao topo e do topo ao fundo seguinte (fechamento diário)
   14/01/2015 US$ 178 -> topo 16/12/2017 US$ 19.497: 109,5 vezes em 1067 dias | -> fundo 15/12/2018 US$ 3.237: -83% em 364 dias
   15/12/2018 US$ 3.237 -> topo 08/11/2021 US$ 67.567: 20,9 vezes em 1059 dias | -> fundo 21/11/2022 US$ 15.787: -77% em 378 dias
   21/11/2022 US$ 15.787 -> topo 06/10/2025 US$ 124.753: 7,9 vezes em 1050 dias | -> fundo 30/06/2026 US$ 58.559: -53% em 267 dias
   Do fundo ao topo: 1.067, 1.059 e 1.050 dias. A alta encolheu de 109,5 para 20,9 e 7,9 vezes. O fundo de 2026 vale até 30/09/2026.


## Parte 19: quanto tempo para recuperar a máxima (Capítulo 15)

In [21]:
print("Parte 19. Dias entre uma máxima histórica de ciclo e o primeiro fechamento acima dela")
for t in TOPOS_C:
    depois = btc.loc[t:]; acima = depois[depois > btc[t]]
    if len(acima):
        print(f"   {t:%d/%m/%Y} (US$ {nr(btc[t])}): superada em {acima.index[0]:%d/%m/%Y}, {(acima.index[0] - t).days} dias depois")
    else:
        print(f"   {t:%d/%m/%Y} (US$ {nr(btc[t])}): ainda não superada; {(btc.index[-1] - t).days} dias depois, o preço está {ps(btc.iloc[-1] / btc[t] - 1)}")
print("   Quem compra no topo de um ciclo pode esperar anos para voltar ao preço de compra.")

Parte 19. Dias entre uma máxima histórica de ciclo e o primeiro fechamento acima dela
   16/12/2017 (US$ 19.497): superada em 30/11/2020, 1080 dias depois
   08/11/2021 (US$ 67.567): superada em 04/03/2024, 847 dias depois
   06/10/2025 (US$ 124.753): ainda não superada; 359 dias depois, o preço está -33%
   Quem compra no topo de um ciclo pode esperar anos para voltar ao preço de compra.


## Parte 20: os sinais nos topos e nos fundos (Capítulo 15)

In [22]:
f30 = fr.resample("D").sum().rolling(30).mean() * 365
PONTOS_C = [("topo de 2017", "2017-11-01", "2018-01-31", "max"), ("fundo de 2018", "2018-11-01", "2019-01-31", "min"),
            ("topo de 2021", "2021-10-15", "2021-11-30", "max"), ("fundo de 2022", "2022-06-01", "2022-12-31", "min"),
            ("topo de 2025", "2025-09-25", "2025-10-20", "max"), ("fundo de 2026", "2026-02-01", "2026-07-15", "min")]
print("Parte 20. Sinais no dia do extremo (IFR e média de 200 semanas pela semana do extremo, como na Parte 4)")
for nome, a, b, tipo in PONTOS_C:
    d = btc.loc[a:b].idxmax() if tipo == "max" else btc.loc[a:b].idxmin()
    w = SEM.loc[a:b]; sw = w.idxmax() if tipo == "max" else w.idxmin()
    mm = SEM19.rolling(200).mean().get(sw, np.nan)
    fin = f"{pc(f30[d], 1)}" if d in f30.index and pd.notna(f30[d]) else "sem dado"
    fol = f"{pc(PCT[d])}" if d in PCT.index else "sem dado"
    print(f"   {nome:<14} {d:%d/%m/%Y}: Mayer {nr(MAYER[d], 2)} | IFR semanal {nr(IFR[sw], 1)} | preço / média de 200 semanas "
          + (f"{nr(SEM[sw] / mm, 2)}" if pd.notna(mm) else "sem dado") + f" | financiamento {fin} | fôlego {fol}")
print("   Os topos ficaram menos eufóricos a cada ciclo; os fundos se parecem mais entre si do que os topos.")

Parte 20. Sinais no dia do extremo (IFR e média de 200 semanas pela semana do extremo, como na Parte 4)
   topo de 2017   16/12/2017: Mayer 3,74 | IFR semanal 90,3 | preço / média de 200 semanas sem dado | financiamento sem dado | fôlego sem dado
   fundo de 2018  15/12/2018: Mayer 0,51 | IFR semanal 28,8 | preço / média de 200 semanas 1,02 | financiamento sem dado | fôlego sem dado
   topo de 2021   08/11/2021: Mayer 1,48 | IFR semanal 68,8 | preço / média de 200 semanas 3,77 | financiamento 32,5% | fôlego 74%
   fundo de 2022  21/11/2022: Mayer 0,71 | IFR semanal 31,4 | preço / média de 200 semanas 0,68 | financiamento -2,0% | fôlego 11%
   topo de 2025   06/10/2025: Mayer 1,18 | IFR semanal 64,2 | preço / média de 200 semanas 2,31 | financiamento 6,0% | fôlego 89%
   fundo de 2026  30/06/2026: Mayer 0,78 | IFR semanal 32,9 | preço / média de 200 semanas 0,95 | financiamento 2,5% | fôlego 11%
   Os topos ficaram menos eufóricos a cada ciclo; os fundos se parecem mais entre si do que 

## Para ir além
1. Na Parte 1, ligue os topos de 2017, 2021 e 2025 em escala log. A linha dos topos também perde inclinação?
2. Na Parte 2, meça o alvo do topo duplo em dias: o tempo entre os topos ajuda a prever quando o fundo vem?
3. Na Parte 3, troque o critério de rompimento por dois fechamentos semanais 3% abaixo da faixa. O rompimento de junho de 2026 ainda conta?
4. Na Parte 5, troque a média de 200 dias pela de 100 e pela de 50. Qual troca mais de lado?
5. Na Parte 6, calcule a distância de 2 ATR em cada ano. O stop de 2026 seria maior ou menor que o de 2017?
6. Na Parte 9, troque o limite de 2,4 pela média dos picos de cada ciclo. Um limite que encolhe teria avisado os topos de 2024 e 2025?
7. Na Parte 11, conte só os gaps do IBIT maiores que 3%. Eles fecham mais ou menos vezes?
8. Na Parte 12, corrija o ticker da UNI para UNI7083-USD. O fôlego muda?
9. Na Parte 13, troque as médias mensais por semanais. As correlações sobem ou descem?
10. Na Parte 16, use a aceleração do M2 (a variação da variação). A relação fica mais estável?
11. Na Parte 18, meça os ciclos pelo fechamento semanal. As durações continuam tão parecidas?
12. Na Parte 20, acrescente a taxa de financiamento de 7 dias. Ela marca melhor os fundos?

**Governança:** os resultados valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo.

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros.

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido.

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.